# Notebook 11: Final tests. Test halves, challenge sets, and the sealed KaraAgro half, opened once

Runs only after `frozen_choices_v3.json` exists (Notebook 10). Nothing is chosen here; everything is scored once with the frozen v3 instance and the frozen Phase 1 instance.

1. **Unseal `d5_test`** and record the time.
2. **Frozen instances on every evaluation set** (LiteRT files, as a phone would run them): Phase 1 internal test (sanity), `d3c_test`, `d3m_test` (exploratory), `d5_test` (confirmatory, unadapted), and the challenge sets (H5): near-unfamiliar cassava, near-unfamiliar maize, far-unfamiliar.
3. **H3 primary confirmatory endpoint.** The Phase 1 maize model is adapted with the registered label-free method and N on images drawn from `d5_adapt` (draw 1; draws 2 and 3 as sensitivity) and scored on `d5_test`. Paired group bootstrap (2,000 resamples) of the macro-F1 difference (MSV, healthy) versus the unadapted Phase 1 model. "Confirmed" requires the 95% interval to exclude zero and the estimate to be ≥ +0.02.
4. **H1 secondary confirmatory.** The six E11 models that held out KaraAgro are scored on `d5_test`: cross-source drop and the shift-augmentation effect, paired by seed.
5. **H5 abstention under shift.** Max-softmax versus energy for predicting correctness (AUROC) on labelled sets, and for separating familiar from unfamiliar inputs; false-acceptance rates on the challenge sets under the global rule, the per-class rule (C1) and the energy rule (C2).

**Runtime.** A100 (the adaptation and Keras scoring) ; about 60–90 minutes. Resumable.

In [ ]:
# 1. Setup, integrity (registration, v2 and v3 freezes, adaptation registration), unseal d5_test
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, math, collections
import numpy as np, pandas as pd
import tensorflow as tf, keras
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'
RUNS = f'{DRIVE}/runs_v3'; OUT = f'{DRIVE}/records/nb11'; PRED = f'{OUT}/predictions'
for d in [OUT, PRED, WORK]: os.makedirs(d, exist_ok=True)
def check(path, sha_path=None):
    sha = hashlib.sha256(open(path, 'rb').read()).hexdigest(); assert sha == open(sha_path or path + '.sha256').read().strip(), f'{path} changed'; return sha
reg_sha = check(f'{DRIVE}/records/protocol_v3_registration.json'); check(f'{DRIVE}/records/protocol_v3_registration_text.md')
f2 = f'{DRIVE}/records/frozen_choices_v2.json'; check(f2, f'{DRIVE}/records/frozen_choices_v2.sha256'); F2 = json.load(open(f2)); DEP1 = F2['deployed_instance']
f3 = f'{DRIVE}/records/frozen_choices_v3.json'; assert os.path.exists(f3), 'Phase 2 is not frozen: run Notebook 10 first'
sha3 = check(f3, f'{DRIVE}/records/frozen_choices_v3.sha256'); F3 = json.load(open(f3)); DEP3 = F3['deployed_instance_v3']
adec_path = f'{DRIVE}/records/v3_adaptation_registered.json'; check(adec_path); ADEC = json.load(open(adec_path))
assert F3['adaptation_registration_sha256'] == open(adec_path + '.sha256').read().strip()
unseal_path = f'{DRIVE}/records/d5_unsealed.json'
if not os.path.exists(unseal_path):
    json.dump({'unsealed_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'frozen_choices_v3_sha256': sha3, 'registration_sha256': reg_sha,
               'note': 'd5_test opened for the first time, after the v3 freeze, by Notebook 11'}, open(unseal_path, 'w'), indent=1)
UNSEAL = json.load(open(unseal_path)); print('d5_test unsealed at', UNSEAL['unsealed_utc'], '| v3 frozen', F3['frozen_utc'], '| registered', json.load(open(f'{DRIVE}/records/protocol_v3_registration.json'))['registered_utc'])
for c in DEP3: print(c, 'v3:', DEP3[c]['model'], DEP3[c]['format'], 'k', DEP3[c]['k_float_layers'], DEP3[c]['MiB'], 'MiB | T', DEP3[c]['temperature_T'], 'thr', DEP3[c]['threshold'], '|', DEP3[c]['abstention'])
print('v3 adaptation:', ADEC['maize'])
try:
    import ai_edge_litert; from ai_edge_litert.interpreter import Interpreter, OpResolverType; litert_pkg = f'ai_edge_litert {getattr(ai_edge_litert, "__version__", "")}'
except Exception:
    Interpreter = tf.lite.Interpreter; OpResolverType = tf.lite.experimental.OpResolverType; litert_pkg = 'tf.lite (bundled with TensorFlow)'
RECIPE = {'head_lr': 1e-3, 'head_max_epochs': 5, 'finetune_lr': 1e-5, 'finetune_max_epochs': 25, 'patience': 5,
          'unfreeze_from': {'MobileNetV2': 'block_13', 'MobileNetV3Small': 'expanded_conv_8', 'EfficientNetB0': 'block6a'}}
BATCH = 32; IMG = 224
gpus = tf.config.list_physical_devices('GPU'); assert gpus, 'Choose an A100 runtime'
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
env = {'notebook': '11_final_tests', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0], 'tensorflow': tf.__version__,
       'keras': keras.__version__, 'numpy': np.__version__, 'litert_interpreter': litert_pkg, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb11.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Manifest v3 (full, d5_test included), working images, evaluation sets
seal = json.load(open(f'{DRIVE}/records/phase2_data_v3.json'))
mpath = f'{DRIVE}/manifests/{seal["manifest_v3"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == seal['sha256'] == F3['manifest_v3_sha256'], 'manifest_v3 changed'
M = pd.read_csv(mpath, dtype=str).fillna('')
for src in ['D1', 'D3', 'D3x', 'D4', 'D5', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}
N_OUT = {c: len(v) for c, v in CLASSES.items()}; LABEL_IDX = {c: {l: i for i, l in enumerate(v)} for c, v in CLASSES.items()}; INV = {c: {i: l for l, i in LABEL_IDX[c].items()} for c in CLASSES}
def rows(part, crop=None, labels=None):
    d = M[M.partition == part]
    if crop: d = d[d.crop == crop]
    if labels: d = d[d.research_label.isin(labels)]
    return d.sort_values('sample_id').reset_index(drop=True)
SETS = {  # name: (crop instance, dataframe, labelled?, status)
  ('cassava', 'internal_test'): (rows('test', 'cassava'), True, 'exploratory (Phase 1 sanity)'),
  ('cassava', 'd3c_test'): (rows('d3c_test', labels=CLASSES['cassava']), True, 'exploratory'),
  ('cassava', 'challenge_near_cassava'): (rows('challenge_near_cassava'), False, 'H5'),
  ('cassava', 'challenge_far'): (rows('challenge_far'), False, 'H5'),
  ('maize', 'internal_test'): (rows('test', 'maize'), True, 'exploratory (Phase 1 sanity)'),
  ('maize', 'd3m_test'): (rows('d3m_test', labels=['M01', 'M05']), True, 'exploratory'),
  ('maize', 'd5_test'): (rows('d5_test', labels=['M01', 'M05']), True, 'CONFIRMATORY'),
  ('maize', 'challenge_near_maize'): (rows('challenge_near_maize'), False, 'H5'),
  ('maize', 'challenge_far'): (rows('challenge_far'), False, 'H5'),
}
EVAL_LABELS = {('cassava', 'internal_test'): [0, 1, 2], ('cassava', 'd3c_test'): [0, 1, 2], ('maize', 'internal_test'): [0, 1, 2, 3, 4], ('maize', 'd3m_test'): [0, 4], ('maize', 'd5_test'): [0, 4]}
for k, (d, lab, st) in SETS.items(): print(k, len(d), 'images', d.group_id.nunique(), 'groups', d.research_label.value_counts().to_dict() if lab else d.original_label.value_counts().to_dict(), '|', st)
ADAPT = {'maize': rows('d5_adapt', labels=['M01', 'M05']), 'cassava': rows('d3c_adapt', labels=CLASSES['cassava'])}
TESTDF = {'maize': SETS[('maize', 'd5_test')][0], 'cassava': SETS[('cassava', 'd3c_test')][0]}; TESTNAME = {'maize': 'd5_test', 'cassava': 'd3c_test'}
EVAL_LABELS_CROP = {'maize': [0, 4], 'cassava': [0, 1, 2]}

In [ ]:
# 3. Data pipeline (same code as Notebooks 07 and 09) (all augmentation applies to training batches only)
def load_img(path, img):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    x = tf.image.resize_with_pad(x, img, img, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(x), 0, 255), tf.uint8)
def base_ds(df, classes, img):
    paths = [f'{WORK}/{p}' for p in df.working_path]; y = [classes.index(l) for l in df.research_label]
    return tf.data.Dataset.from_tensor_slices((paths, y)).map(lambda p, l: (load_img(p, img), l), num_parallel_calls=tf.data.AUTOTUNE).cache()

def phase1_aug(seed):
    return keras.Sequential([keras.layers.RandomFlip('horizontal_and_vertical', seed=seed), keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=seed),
                             keras.layers.RandomBrightness(0.1, value_range=(0, 255), seed=seed), keras.layers.RandomContrast(0.1, seed=seed)])

def random_resized_crop(x, img):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.5, 1.0); ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.75), tf.math.log(4 / 3)))
    w = tf.minimum(tf.sqrt(area * ratio), 1.0); h = tf.minimum(tf.sqrt(area / ratio), 1.0)
    x1 = tf.random.uniform([B]) * (1 - w); y1 = tf.random.uniform([B]) * (1 - h)
    boxes = tf.stack([y1, x1, y1 + h, x1 + w], axis=1)
    return tf.image.crop_and_resize(x, boxes, tf.range(B), [img, img])
def colour_jitter(x):
    B = tf.shape(x)[0]; sh = [B, 1, 1, 1]
    x = x + tf.random.uniform(sh, -0.25, 0.25) * 255.0                                           # brightness
    m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True); x = m + (x - m) * tf.random.uniform(sh, 0.75, 1.25)   # contrast
    g = tf.image.rgb_to_grayscale(x); x = g + (x - g) * tf.random.uniform(sh, 0.75, 1.25)                     # saturation
    x = tf.clip_by_value(x, 0, 255)
    x = tf.map_fn(lambda im: tf.image.random_hue(im / 255.0, 0.05) * 255.0, x)                               # hue, per image
    return tf.clip_by_value(x, 0, 255)
def random_erasing(x, img, p=0.5, max_area=0.25):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.02, max_area) * img * img; ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.3), tf.math.log(3.3)))
    h = tf.minimum(tf.cast(tf.sqrt(area * ratio), tf.int32), img); w = tf.minimum(tf.cast(tf.sqrt(area / ratio), tf.int32), img)
    y1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - h, tf.float32), tf.int32); x1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - w, tf.float32), tf.int32)
    ys = tf.range(img)[None, :, None]; xs = tf.range(img)[None, None, :]
    inside = (ys >= y1[:, None, None]) & (ys < (y1 + h)[:, None, None]) & (xs >= x1[:, None, None]) & (xs < (x1 + w)[:, None, None])
    apply = tf.random.uniform([B]) < p
    mask = inside & apply[:, None, None]
    return tf.where(mask[..., None], 127.5, x)
def cutmix(x, y, sw, img, p=0.5):
    B = tf.shape(x)[0]; lam = tf.random.uniform([])
    cut = tf.sqrt(1.0 - lam); ch = tf.cast(cut * img, tf.int32); cw = ch
    cy = tf.random.uniform([], 0, img, dtype=tf.int32); cx = tf.random.uniform([], 0, img, dtype=tf.int32)
    y1 = tf.clip_by_value(cy - ch // 2, 0, img); y2 = tf.clip_by_value(cy + ch // 2, 0, img); x1 = tf.clip_by_value(cx - cw // 2, 0, img); x2 = tf.clip_by_value(cx + cw // 2, 0, img)
    ys = tf.range(img)[:, None]; xs = tf.range(img)[None, :]
    box = ((ys >= y1) & (ys < y2) & (xs >= x1) & (xs < x2))[None, :, :, None]
    xr, yr, sr = tf.reverse(x, [0]), tf.reverse(y, [0]), tf.reverse(sw, [0])
    lam_adj = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / float(img * img)
    xm = tf.where(box, xr, x); ym = lam_adj * y + (1 - lam_adj) * yr; sm = lam_adj * sw + (1 - lam_adj) * sr
    do = tf.random.uniform([]) < p
    return tf.cond(do, lambda: (xm, ym, sm), lambda: (x, y, sw))

def train_ds(base, n_classes, seed, weights, img, aug_kind):
    w = tf.constant(weights, tf.float32); p1 = phase1_aug(seed)
    ds = base.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes), tf.gather(w, l)), num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    if aug_kind == 'shift':
        def shift(x, y, sw):
            x = random_resized_crop(x, img); x = colour_jitter(x); x = random_erasing(x, img)
            x, y, sw = cutmix(x, y, sw, img)
            return tf.clip_by_value(p1(x, training=True), 0, 255), y, sw
        ds = ds.map(shift, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y, sw: (tf.clip_by_value(p1(x, training=True), 0, 255), y, sw), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)
def eval_ds(base, n_classes):
    return base.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes))).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 4. Models (as Phase 1, with a variable input size) and the macro-F1 stopper
def build_model(arch, n_classes, img):
    inp = keras.Input((img, img, 3), name='image_0_255')
    if arch == 'MobileNetV2':
        x = keras.layers.Rescaling(1 / 127.5, offset=-1, name='mnv2_scale')(inp)
        base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    else:
        x = inp; base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    base.trainable = False
    x = base(x, training=False); x = keras.layers.GlobalAveragePooling2D(name='gap')(x); x = keras.layers.Dropout(0.2, name='dropout')(x)
    return keras.Model(inp, keras.layers.Dense(n_classes, activation='softmax', name='probs')(x), name=f'{arch}_{n_classes}cls_{img}'), base
def unfreeze_final_blocks(base, arch):
    prefix = RECIPE['unfreeze_from'][arch]; names = [l.name for l in base.layers]
    idx = next((i for i, n in enumerate(names) if n.startswith(prefix)), int(len(names) * 0.75))
    base.trainable = True
    for l in base.layers[:idx]: l.trainable = False
    return {'first_trainable_layer': names[idx], 'trainable_backbone_layers': len(names) - idx, 'total_backbone_layers': len(names)}
class MacroF1Stopper(keras.callbacks.Callback):
    def __init__(self, sel_ds, y_sel, patience, log, phase):
        super().__init__(); self.sel_ds, self.y_sel, self.patience, self.log, self.phase = sel_ds, y_sel, patience, log, phase
        self.best, self.best_w, self.wait, self.t0 = -1.0, None, 0, None
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.sel_ds, verbose=0).argmax(1); f1 = f1_score(self.y_sel, p, average='macro'); acc = accuracy_score(self.y_sel, p)
        self.log.append({'phase': self.phase, 'epoch': epoch + 1, 'train_loss': float(logs.get('loss', np.nan)), 'select_macro_f1': float(f1), 'select_accuracy': float(acc), 'seconds': round(time.time() - self.t0, 1)})
        print(f'  {self.phase} epoch {epoch+1}: loss {logs.get("loss", 0):.4f}  selection macro-F1 {f1:.4f}  acc {acc:.4f}  ({self.log[-1]["seconds"]}s)')
        if f1 > self.best + 1e-6: self.best, self.best_w, self.wait = f1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience: self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

In [ ]:
# ---- OCDF F4 calibration, abstention and selection helpers (Notebook 04) ----
import numpy as np

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T=1.0):
    p = softmax(z, T)
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))

def fit_temperature(z, y, lo=0.05, hi=20.0):
    """Calibration: positive T minimising negative log-likelihood on the calibration partition (Guo et al., 2017).
    Golden-section search on log T (NLL is unimodal in T for fixed logits)."""
    a, b = np.log(lo), np.log(hi); g = (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll(z, y, np.exp(c)), nll(z, y, np.exp(d))
    for _ in range(80):
        if fc < fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = nll(z, y, np.exp(c))
        else: a, c, fc = c, d, fd; d = a + g * (b - a); fd = nll(z, y, np.exp(d))
    T = float(np.exp((a + b) / 2))
    return {'T': round(T, 4), 'nll_before': round(nll(z, y, 1.0), 4), 'nll_after': round(nll(z, y, T), 4)}

def brier(p, y):
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def ece(p, y, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[0]) & (conf <= edges[1])
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def threshold_table(p, y, groups, thresholds=None):
    """Coverage, selective accuracy and accepted groups for each threshold (see docs/METHODS.md)."""
    thresholds = np.round(np.arange(0.50, 0.995, 0.01), 2) if thresholds is None else thresholds
    conf = p.max(1); pred = p.argmax(1); groups = np.asarray(groups); rows = []
    for t in thresholds:
        acc_mask = conf >= t; n_acc = int(acc_mask.sum())
        rows.append({'threshold': float(t), 'coverage': n_acc / len(y),
                     'selective_accuracy': float((pred[acc_mask] == y[acc_mask]).mean()) if n_acc else None,
                     'selective_risk': float((pred[acc_mask] != y[acc_mask]).mean()) if n_acc else None,
                     'correct_accepted_yield': float(((pred == y) & acc_mask).mean()),
                     'accepted_groups': int(len(set(groups[acc_mask]))), 'accepted': n_acc})
    return rows

def choose_threshold(rows, min_acc=0.95, min_cov=0.50, min_groups=50, fallback=0.90):
    """Highest-coverage threshold with selective accuracy >= 0.95, coverage >= 0.50 and >= 50 accepted groups;
    otherwise 'target infeasible' and the predeclared 0.90 for description only."""
    ok = [r for r in rows if r['selective_accuracy'] is not None and r['selective_accuracy'] >= min_acc
          and r['coverage'] >= min_cov and r['accepted_groups'] >= min_groups]
    if ok:
        best = max(ok, key=lambda r: (r['coverage'], -r['threshold']))
        return {'threshold': best['threshold'], 'status': 'target met', **{k: best[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}
    fb = min(rows, key=lambda r: abs(r['threshold'] - fallback))
    return {'threshold': fallback, 'status': 'target infeasible (0.90 used for description only)',
            **{k: fb[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}

def per_class_coverage(p, y, t, classes):
    conf = p.max(1); out = {}
    for i, c in enumerate(classes):
        m = y == i
        out[c] = round(float((conf[m] >= t).mean()), 4) if m.any() else None
    return out

def select_architecture(agg_rows, size_bytes=None, margin=0.01):
    """Pre-declared selection rule: highest mean selection macro-F1; within `margin` of the best, the smaller converted file."""
    best = max(agg_rows, key=lambda r: r['mean_macro_f1'])
    close = [r for r in agg_rows if best['mean_macro_f1'] - r['mean_macro_f1'] <= margin + 1e-12]
    if len(close) > 1 and size_bytes:
        pick = min(close, key=lambda r: size_bytes[r['architecture']])
        return pick['architecture'], f'within {margin} of best; smallest FP32 file chosen'
    return best['architecture'], (f"highest mean selection macro-F1 ({best['mean_macro_f1']:.4f}); "
                                   f"no other architecture within {margin}")
# ---- OCDF F6 evaluation helpers (Notebook 05): metrics, abstention, group bootstrap ----
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def full_metrics(p, y, eval_labels, n_out, threshold=None, groups=None):
    """Evaluation metrics (docs/METHODS.md). `eval_labels`: class indices present in this evaluation set (others are 'not evaluated').
    Predictions keep all n_out outputs (never masked), so predicting an absent class counts as an error."""
    pred = p.argmax(1); L = list(eval_labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    out = {'n_images': int(len(y)), 'n_groups': int(len(set(groups))) if groups is not None else None,
           'accuracy': float(accuracy_score(y, pred)),
           'balanced_accuracy': float(np.mean([np.mean(pred[y == c] == c) for c in L])),
           'macro_f1': float(f1_score(y, pred, labels=L, average='macro', zero_division=0)),
           'per_class': {int(c): {'precision': float(a), 'recall': float(b), 'f1': float(d), 'support': int(s)}
                         for c, a, b, d, s in zip(L, pr, rc, f1, sup)},
           'confusion': confusion_matrix(y, pred, labels=list(range(n_out))).tolist(),
           'nll': float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))),
           'brier': float(np.mean(np.sum((p - np.eye(n_out)[y]) ** 2, axis=1))),
           'ece_15': ece15(p, y)}
    if threshold is not None:
        acc = p.max(1) >= threshold; n = int(acc.sum())
        out.update({'threshold': float(threshold), 'coverage': float(acc.mean()),
                    'selective_accuracy': float((pred[acc] == y[acc]).mean()) if n else None,
                    'selective_risk': float((pred[acc] != y[acc]).mean()) if n else None,
                    'correct_accepted_yield': float(((pred == y) & acc).mean()),
                    'coverage_per_class': {int(c): float(acc[y == c].mean()) for c in L}})
    return out

def ece15(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float); e = 0.0
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fast_accuracy(y, pred):
    return float(np.mean(y == pred))

def fast_macro_f1(y, pred, labels, n_out):
    """Macro-F1 over `labels` from a bincount confusion matrix; identical to sklearn f1_score(average='macro', zero_division=0)."""
    cm = np.bincount(y * n_out + pred, minlength=n_out * n_out).reshape(n_out, n_out)
    tp = np.diag(cm)[labels].astype(float); fp = cm[:, labels].sum(0) - tp; fn = cm[labels, :].sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)))

def group_bootstrap(stat_fn, groups, n_boot=2000, seed=2026):
    """95% percentile interval of stat_fn(index_array), resampling whole groups with replacement (Efron & Tibshirani, 1994)."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind='stable'); lengths = np.bincount(inv); starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq)); lens = lengths[pick]
        offs = np.repeat(starts[pick] - np.cumsum(lens) + lens, lens) + np.arange(lens.sum())
        vals[b] = stat_fn(order[offs])
    return {'estimate': float(stat_fn(np.arange(len(groups)))), 'ci95': [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))],
            'n_boot': n_boot, 'n_groups': int(len(uniq))}

def reliability_bins(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    idx = np.minimum((conf * bins).astype(int), bins - 1); rows = []
    for b in range(bins):
        m = idx == b
        rows.append({'bin_low': b / bins, 'bin_high': (b + 1) / bins, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()) if m.any() else None, 'accuracy': float(correct[m].mean()) if m.any() else None})
    return rows

def risk_coverage(p, y):
    order = np.argsort(-p.max(1)); correct = (p.argmax(1) == y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k
# 5. Adaptation methods (same code as Notebook 09) (label-free: AdaBN, Tent, gated self-training; labelled few-shot reference), evaluation on the test halves
def temperature(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def predict_probs(model, X, bs=64):
    ds = tf.data.Dataset.from_tensor_slices(X).map(lambda x: tf.cast(x, tf.float32)).batch(bs).prefetch(tf.data.AUTOTUNE)
    return model.predict(ds, verbose=0)
def load_array(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(lambda p: load_img(p, IMG), num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
def fresh(path):
    keras.backend.clear_session(); gc.collect(); return keras.models.load_model(path)
def get_base(model): return [l for l in model.layers if isinstance(l, keras.Model)][0]
def bn_layers(base): return [l for l in base.layers if isinstance(l, keras.layers.BatchNormalization)]

def adabn(model, X, bs=32):
    """Adaptive batch normalisation (Li et al., 2018): the backbone's BN moving statistics are replaced by statistics of the
    target images, as a running average over batches (momentum 1 - 1/k on the k-th batch). No weight is changed."""
    base = get_base(model); bns = bn_layers(base)
    for bn in bns: bn.trainable = True                      # Keras updates moving statistics only when the layer is trainable
    before = [np.array(bn.moving_mean) for bn in bns[:3]]
    bs = min(bs, len(X)); k = 0
    for s in range(0, len(X), bs):
        k += 1
        for bn in bns: bn.momentum = 1.0 - 1.0 / k
        base(tf.cast(X[s:s + bs], tf.float32), training=True)
    for bn in bns: bn.momentum = 0.99
    shift = float(np.mean([np.abs(np.array(bn.moving_mean) - b).mean() for bn, b in zip(bns[:3], before)]))
    assert shift > 0, 'AdaBN did not change the BN statistics (check trainable flags / training=True call)'
    return {'bn_layers': len(bns), 'batches': k, 'mean_abs_shift_first3_bn': shift}

def entropy_loss(y_true, y_pred):
    return -tf.reduce_sum(y_pred * tf.math.log(y_pred + 1e-8), axis=-1)

def tent(model, X, n_out, epochs=5, lr=1e-3, seed=0):
    """Tent (Wang et al., 2021): after re-estimating the BN statistics, the BN scale and shift parameters are trained to minimise
    the entropy of the model's predictions on the target images. Nothing else is trainable."""
    info = adabn(model, X)
    base = get_base(model)
    for l in model.layers: l.trainable = False
    base.trainable = True
    for l in base.layers: l.trainable = isinstance(l, keras.layers.BatchNormalization)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=entropy_loss)
    ds = (tf.data.Dataset.from_tensor_slices((X, np.zeros((len(X), n_out), np.float32))).shuffle(1024, seed=seed, reshuffle_each_iteration=True)
          .map(lambda x, y: (tf.cast(x, tf.float32), y)).batch(min(32, len(X))).prefetch(tf.data.AUTOTUNE))
    h = model.fit(ds, epochs=epochs, verbose=0)
    info.update({'epochs': epochs, 'lr': lr, 'entropy_first': float(h.history['loss'][0]), 'entropy_last': float(h.history['loss'][-1]),
                 'trainable_params': int(sum(np.prod(v.shape) for v in model.trainable_variables))})
    return info

def finetune(model, X, y, n_out, epochs, lr=1e-4, seed=0):
    """Phase 1 fine-tuning convention: head plus the final backbone blocks trainable, BN statistics fixed, Phase 1 augmentation."""
    base = get_base(model)
    for l in model.layers: l.trainable = False
    model.get_layer('probs').trainable = True
    ufz = unfreeze_final_blocks(base, 'EfficientNetB0')
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='categorical_crossentropy')
    ds = train_ds(tf.data.Dataset.from_tensor_slices((X, np.asarray(y, np.int64))), n_out, seed, np.ones(n_out, np.float32), IMG, 'phase1')
    h = model.fit(ds, epochs=epochs, verbose=0)
    return {'epochs': epochs, 'lr': lr, 'loss_first': float(h.history['loss'][0]), 'loss_last': float(h.history['loss'][-1]), 'first_trainable_layer': ufz['first_trainable_layer']}

def self_training(model, X, y_true, T, thr, n_out, rounds=2, epochs=10, seed=0):
    """Gated self-training: the frozen abstention rule (temperature T, threshold thr) decides which target images the model may
    label itself; accepted images and their pseudo-labels fine-tune the final blocks; two rounds. y_true is used only to report
    how accurate the pseudo-labels were (never for training)."""
    info = []
    for r in range(rounds):
        p = temperature(predict_probs(model, X), T); conf = p.max(1); acc = conf >= thr; pl = p.argmax(1)
        rec = {'round': r + 1, 'accepted': int(acc.sum()), 'of': int(len(X)),
               'pseudo_label_accuracy_on_accepted': float((pl[acc] == y_true[acc]).mean()) if acc.any() else None,
               'accepted_label_counts': {int(c): int(n) for c, n in zip(*np.unique(pl[acc], return_counts=True))}}
        if acc.sum() < 2:
            rec['action'] = 'stopped: fewer than two accepted images'; info.append(rec); break
        rec['finetune'] = finetune(model, X[acc], pl[acc], n_out, epochs=epochs, seed=seed + r); info.append(rec)
    return info

def evaluate(model, crop, T, thr):
    p = temperature(predict_probs(model, TESTX[crop]), T); y = TESTY[crop]; labels = EVAL_LABELS_CROP[crop]; n_out = N_OUT[crop]
    pred = p.argmax(1); conf = p.max(1); acc = conf >= thr
    return {'macro_f1': fast_macro_f1(y, pred, labels, n_out), 'accuracy': float((pred == y).mean()), 'coverage': float(acc.mean()),
            'selective_accuracy': float((pred[acc] == y[acc]).mean()) if acc.any() else None, 'mean_confidence': float(conf.mean()),
            'recall': {INV[crop][l]: float((pred[y == l] == l).mean()) for l in labels}}, pd.DataFrame({'y': y, 'pred': pred, 'conf': conf})

def draw_adapt(crop, N, draw):
    """N images drawn group-wise (whole capture groups, random order) from the crop's adapt half; the last group is cut to reach N."""
    d = ADAPT[crop]; rng = np.random.default_rng(2026 * 1000 + N * 10 + draw)
    gids = d.group_id.unique().copy(); rng.shuffle(gids); rows = []
    for g in gids:
        rows.append(d[d.group_id == g])
        if sum(len(r) for r in rows) >= N: break
    sel = pd.concat(rows).iloc[:N]
    return load_array(sel), np.array([LABEL_IDX[crop][l] for l in sel.research_label]), sel

In [ ]:
# 6. Frozen LiteRT instances (v3 and Phase 1) scored on every evaluation set; metrics with group bootstrap
def make_interpreter(b, threads, xnnpack=True):
    it = Interpreter(model_content=b, num_threads=threads) if xnnpack else Interpreter(model_content=b, num_threads=threads, experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
    it.allocate_tensors(); return it
def run_tflite(b, X, batch=32):
    for xnn in [True, False]:
        try:
            it = make_interpreter(b, os.cpu_count(), xnn); i, o = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
            it.resize_tensor_input(i, [batch, IMG, IMG, 3]); it.allocate_tensors(); out = []
            for s in range(0, len(X), batch):
                xb = X[s:s + batch].astype(np.float32); n = len(xb)
                if n < batch: xb = np.concatenate([xb, np.zeros((batch - n, IMG, IMG, 3), np.float32)])
                it.set_tensor(i, xb); it.invoke(); out.append(it.get_tensor(o)[:n].copy())
            return np.concatenate(out)
        except Exception as e:
            if xnn: print('  XNNPACK failed (', type(e).__name__, '); using builtin kernels')
            else: raise
def energy(z, T): return -T * np.log(np.exp(z / T - (z / T).max(1, keepdims=True)).sum(1)) - (z / T).max(1) * T
INST = {}
for crop in CLASSES:
    d1 = DEP1[crop]; d3 = DEP3[crop]
    INST[(crop, 'phase1')] = {'file': d1['file'], 'T': float(d1['temperature_T']), 'thr': float(d1['threshold']), 'label': f'Phase 1 {d1["format"]}', 'C1': None, 'E95': None}
    INST[(crop, 'v3')] = {'file': d3['file'], 'T': float(d3['temperature_T']), 'thr': float(d3['threshold']), 'label': f'v3 {d3["model"]} {d3["format"]} k{d3["k_float_layers"]}',
                          'C1': {LABEL_IDX[crop][c]: v['threshold'] for c, v in d3['C1_per_class'].items() if 'threshold' in v} if d3['abstention'].startswith('C1') else None, 'E95': d3['C2_energy_p95']}
    for k in ['phase1', 'v3']:
        b = open(f'{DRIVE}/{INST[(crop, k)]["file"]}', 'rb').read(); assert hashlib.sha256(b).hexdigest() == (d1 if k == 'phase1' else d3)['sha256'], 'deployed file changed'
X_CACHE = {}
def get_X(key):
    if key not in X_CACHE: X_CACHE[key] = load_array(SETS[key][0])
    return X_CACHE[key]
LOG = {}                                  # (crop, inst, set) -> logits
for (crop, sname), (d, labelled, status) in SETS.items():
    for inst in ['phase1', 'v3']:
        p = f'{PRED}/logits_{crop}_{inst}_{sname}.npy'
        if not os.path.exists(p):
            b = open(f'{DRIVE}/{INST[(crop, inst)]["file"]}', 'rb').read(); t0 = time.time(); z = run_tflite(b, get_X((crop, sname))); np.save(p, z)
            print(f'{crop} {inst} {sname}: {len(z)} images in {time.time() - t0:.0f}s')
        LOG[(crop, inst, sname)] = np.load(p)
RESULTS = {'unsealed': UNSEAL, 'frozen_v3_sha256': sha3, 'labelled_sets': {}, 'challenge_sets': {}}
TAB = []
for (crop, sname), (d, labelled, status) in SETS.items():
    for inst in ['phase1', 'v3']:
        I = INST[(crop, inst)]; z = LOG[(crop, inst, sname)]; p = softmax(z, I['T']); pred = p.argmax(1); conf = p.max(1); groups = d.group_id.values
        acc_g = conf >= I['thr']
        acc_c1 = np.array([c >= I['C1'].get(int(k), I['thr']) for c, k in zip(conf, pred)]) if I['C1'] else None
        e = energy(z, I['T']); acc_e = e <= I['E95'] if I['E95'] is not None else None
        pd.DataFrame({'sample_id': d.sample_id, 'group_id': groups, 'label': d.research_label if labelled else d.original_label, 'pred': pred, 'conf': conf, 'energy': e,
                      'accept_global': acc_g, 'accept_c1': acc_c1 if acc_c1 is not None else '', 'accept_energy': acc_e if acc_e is not None else ''}).to_csv(f'{PRED}/pred_{crop}_{inst}_{sname}.csv', index=False)
        if labelled:
            y = np.array([LABEL_IDX[crop][l] for l in d.research_label]); labels = EVAL_LABELS[(crop, sname)]; n_out = N_OUT[crop]
            fm = full_metrics(p, y, labels, n_out, threshold=I['thr'], groups=groups)
            r = {'status': status, 'instance': I['label'], 'n': len(y), 'groups': int(len(set(groups))),
                 'macro_f1': group_bootstrap(lambda ix: fast_macro_f1(y[ix], pred[ix], labels, n_out), groups), 'accuracy': group_bootstrap(lambda ix: fast_accuracy(y[ix], pred[ix]), groups),
                 'per_class_recall': {INV[crop][l]: fm['per_class'][l]['recall'] for l in labels}, 'ece_15': fm['ece_15'], 'brier': fm['brier'], 'nll': fm['nll'],
                 'coverage_global': float(acc_g.mean()), 'selective_accuracy_global': fm['selective_accuracy'],
                 'coverage_c1': float(acc_c1.mean()) if acc_c1 is not None else None, 'selective_accuracy_c1': float((pred[acc_c1] == y[acc_c1]).mean()) if acc_c1 is not None and acc_c1.any() else None,
                 'auroc_correctness_msp': float(roc_auc_score((pred == y).astype(int), conf)) if 0 < (pred == y).mean() < 1 else None,
                 'auroc_correctness_energy': float(roc_auc_score((pred == y).astype(int), -e)) if 0 < (pred == y).mean() < 1 else None}
            RESULTS['labelled_sets'][f'{crop}/{sname}/{inst}'] = r
            TAB.append({'crop': crop, 'set': sname, 'status': status, 'instance': inst, 'n': len(y), 'macro_f1': r['macro_f1']['estimate'], 'f1_lo': r['macro_f1']['ci95'][0], 'f1_hi': r['macro_f1']['ci95'][1],
                        'accuracy': r['accuracy']['estimate'], 'coverage': r['coverage_global'], 'sel_acc': r['selective_accuracy_global'], 'ece': r['ece_15'],
                        'auroc_msp': r['auroc_correctness_msp'], 'auroc_energy': r['auroc_correctness_energy'], **{f'recall_{k}': v for k, v in r['per_class_recall'].items()}})
        else:
            r = {'status': status, 'instance': I['label'], 'n': len(d), 'false_acceptance_global': float(acc_g.mean()),
                 'false_acceptance_c1': float(acc_c1.mean()) if acc_c1 is not None else None, 'false_acceptance_energy': float(acc_e.mean()) if acc_e is not None else None,
                 'mean_confidence': float(conf.mean()), 'predicted_class_counts': {INV[crop][int(k)]: int(v) for k, v in zip(*np.unique(pred, return_counts=True))},
                 'by_original_label': {lab: {'n': int((d.original_label == lab).sum()), 'false_acceptance_global': float(acc_g[(d.original_label == lab).values].mean())} for lab in sorted(d.original_label.unique())}}
            RESULTS['challenge_sets'][f'{crop}/{sname}/{inst}'] = r
T1 = pd.DataFrame(TAB); T1.to_csv(f'{OUT}/final_labelled_sets.csv', index=False)
pd.set_option('display.width', 250); print(T1.round(4).to_string(index=False))
print('\nchallenge sets (false-acceptance = share of unfamiliar images NOT rejected):')
print(pd.DataFrame([{'crop': k.split('/')[0], 'set': k.split('/')[1], 'instance': k.split('/')[2], **{kk: vv for kk, vv in v.items() if kk in ('n', 'false_acceptance_global', 'false_acceptance_c1', 'false_acceptance_energy', 'mean_confidence')}} for k, v in RESULTS['challenge_sets'].items()]).round(4).to_string(index=False))

In [ ]:
# 7. H5: familiar vs unfamiliar separation (AUROC of max-softmax and energy), per instance and crop
H5 = {}
for crop in CLASSES:
    fam = [s for (c, s), (d, lab, st) in SETS.items() if c == crop and lab and s != 'internal_test']
    unf = [s for (c, s), (d, lab, st) in SETS.items() if c == crop and not lab]
    for inst in ['phase1', 'v3']:
        I = INST[(crop, inst)]
        zf = np.concatenate([LOG[(crop, inst, s)] for s in fam]); zu = np.concatenate([LOG[(crop, inst, s)] for s in unf])
        for uname, zuu in [('all_unfamiliar', zu)] + [(s, LOG[(crop, inst, s)]) for s in unf]:
            yb = np.r_[np.ones(len(zf)), np.zeros(len(zuu))]
            msp = np.r_[softmax(zf, I['T']).max(1), softmax(zuu, I['T']).max(1)]; en = np.r_[-energy(zf, I['T']), -energy(zuu, I['T'])]
            H5[f'{crop}/{inst}/{uname}'] = {'familiar_sets': fam, 'n_familiar': int(len(zf)), 'n_unfamiliar': int(len(zuu)),
                                             'auroc_msp': float(roc_auc_score(yb, msp)), 'auroc_energy': float(roc_auc_score(yb, en))}
RESULTS['h5_familiar_vs_unfamiliar'] = H5
print(pd.DataFrame([{'key': k, **{kk: v[kk] for kk in ('n_familiar', 'n_unfamiliar', 'auroc_msp', 'auroc_energy')}} for k, v in H5.items()]).round(4).to_string(index=False))

In [ ]:
# 8. H3 PRIMARY CONFIRMATORY ENDPOINT: registered adaptation applied once to the Phase 1 maize model on d5_adapt, scored on d5_test (paired group bootstrap)
STARTS = {'maize': {'phase1': {'path': f'{DRIVE}/runs/maize_EfficientNetB0_s42/best_model.keras', 'T': float(DEP1['maize']['temperature_T']), 'threshold': float(DEP1['maize']['threshold'])}},
          'cassava': {'phase1': {'path': f'{DRIVE}/runs/cassava_EfficientNetB0_s42/best_model.keras', 'T': float(DEP1['cassava']['temperature_T']), 'threshold': float(DEP1['cassava']['threshold'])}}}
TESTX = {'maize': get_X(('maize', 'd5_test')), 'cassava': get_X(('cassava', 'd3c_test'))}
TESTY = {c: np.array([LABEL_IDX[c][l] for l in TESTDF[c].research_label]) for c in TESTX}
method, N = ADEC['maize']['method'], int(ADEC['maize']['N'])
def apply_registered(model, crop, X, y, T, thr, seed):
    if method == 'adabn': return adabn(model, X)
    if method == 'tent': return tent(model, X, N_OUT[crop], seed=seed)
    if method == 'self_training': return self_training(model, X, y, T, thr, N_OUT[crop], seed=seed)
    raise ValueError(method)
H3 = {'registered': ADEC['maize'], 'rule': 'paired group bootstrap (2,000 resamples) of macro-F1 (M01, M05) on d5_test, adapted minus unadapted Phase 1 model; confirmed if the 95% CI excludes 0 and the estimate >= +0.02', 'draws': {}}
for crop in ['maize', 'cassava']:
    S = STARTS[crop]['phase1']; T, thr = S['T'], S['threshold']; y = TESTY[crop]; groups = TESTDF[crop].group_id.values; labels = EVAL_LABELS_CROP[crop]; n_out = N_OUT[crop]
    p0_path = f'{PRED}/h3_{crop}_unadapted_probs.npy'
    if not os.path.exists(p0_path): m0 = fresh(S['path']); np.save(p0_path, temperature(predict_probs(m0, TESTX[crop]), T)); del m0
    p0 = np.load(p0_path); pred0 = p0.argmax(1)
    for draw in [1, 2, 3]:
        key = f'{crop}_d{draw}'; rp = f'{OUT}/h3_{key}.json'
        if os.path.exists(rp): H3['draws'][key] = json.load(open(rp)); continue
        m = fresh(S['path']); X, ya, sel = draw_adapt(crop, N, draw); t0 = time.time()
        detail = apply_registered(m, crop, X, ya, T, thr, draw)
        p1 = temperature(predict_probs(m, TESTX[crop]), T); pred1 = p1.argmax(1); np.save(f'{PRED}/h3_{key}_adapted_probs.npy', p1)
        diff = group_bootstrap(lambda ix: fast_macro_f1(y[ix], pred1[ix], labels, n_out) - fast_macro_f1(y[ix], pred0[ix], labels, n_out), groups)
        rec = {'crop': crop, 'test_set': TESTNAME[crop], 'draw': draw, 'adapt_source': 'd5_adapt' if crop == 'maize' else 'd3c_adapt', 'N': N, 'method': method,
               'adapt_groups': int(sel.group_id.nunique()), 'adapt_label_counts': sel.research_label.value_counts().to_dict(), 'detail': detail,
               'unadapted': {'macro_f1': fast_macro_f1(y, pred0, labels, n_out), 'accuracy': fast_accuracy(y, pred0), 'recall': {INV[crop][l]: float((pred0[y == l] == l).mean()) for l in labels}, 'coverage': float((p0.max(1) >= thr).mean())},
               'adapted': {'macro_f1': fast_macro_f1(y, pred1, labels, n_out), 'accuracy': fast_accuracy(y, pred1), 'recall': {INV[crop][l]: float((pred1[y == l] == l).mean()) for l in labels}, 'coverage': float((p1.max(1) >= thr).mean())},
               'paired_difference_macro_f1': diff, 'confirmed': bool(diff['ci95'][0] > 0 and diff['estimate'] >= 0.02),
               'status': ('PRIMARY CONFIRMATORY' if (crop == 'maize' and draw == 1) else 'sensitivity' if crop == 'maize' else 'secondary (exploratory)'), 'minutes': round((time.time() - t0) / 60, 2)}
        json.dump(rec, open(rp, 'w'), indent=1, default=float); H3['draws'][key] = rec; del m; gc.collect()
    for draw in [1, 2, 3]:
        r = H3['draws'][f'{crop}_d{draw}']
        print(f'{crop} draw {draw} [{r["status"]}]: unadapted {r["unadapted"]["macro_f1"]:.4f} -> adapted {r["adapted"]["macro_f1"]:.4f}; '
              f'difference {r["paired_difference_macro_f1"]["estimate"]:+.4f} CI [{r["paired_difference_macro_f1"]["ci95"][0]:+.4f}, {r["paired_difference_macro_f1"]["ci95"][1]:+.4f}] -> {"CONFIRMED" if r["confirmed"] else "not confirmed"}')
prim = H3['draws']['maize_d1']; H3['primary_result'] = {'confirmed': prim['confirmed'], 'estimate': prim['paired_difference_macro_f1']['estimate'], 'ci95': prim['paired_difference_macro_f1']['ci95']}
RESULTS['h3_primary'] = H3; print('\nPRIMARY ENDPOINT:', json.dumps(H3['primary_result']))

In [ ]:
# 9. H1 secondary confirmatory: the six E11 models that held out KaraAgro, scored on d5_test (2-class models: MSV vs healthy)
MAIZE2 = ['M01', 'M05']; d5 = SETS[('maize', 'd5_test')][0]; y2 = np.array([MAIZE2.index(l) for l in d5.research_label]); groups = d5.group_id.values
H1 = {'rule': 'E11 fold holding out GH-K (KaraAgro): in-source selection macro-F1 vs d5_test macro-F1; shift minus Phase 1 augmentation paired by seed', 'runs': {}}
for arm in ['phase1', 'shift']:
    for seed in [17, 42, 2026]:
        rid = f'E11_maize_holdout-GH-K_{arm}_s{seed}'; rp = f'{OUT}/h1_{rid}.json'
        if os.path.exists(rp): H1['runs'][rid] = json.load(open(rp)); continue
        done = json.load(open(f'{RUNS}/{rid}/done.json')); m = fresh(f'{RUNS}/{rid}/best_model.keras')
        p = predict_probs(m, TESTX['maize']); pred = p.argmax(1); np.save(f'{PRED}/h1_{rid}_probs.npy', p)
        rec = {'run_id': rid, 'arm': arm, 'seed': seed, 'in_source_select_macro_f1': done['in_source_select_macro_f1'],
               'd5_test_macro_f1': group_bootstrap(lambda ix: fast_macro_f1(y2[ix], pred[ix], [0, 1], 2), groups), 'd5_test_accuracy': fast_accuracy(y2, pred),
               'recall': {c: float((pred[y2 == i] == i).mean()) for i, c in enumerate(MAIZE2)}}
        rec['drop_macro_f1'] = rec['in_source_select_macro_f1'] - rec['d5_test_macro_f1']['estimate']
        json.dump(rec, open(rp, 'w'), indent=1, default=float); H1['runs'][rid] = rec; del m; gc.collect()
H1['summary'] = {}
for arm in ['phase1', 'shift']:
    rs = [H1['runs'][f'E11_maize_holdout-GH-K_{arm}_s{s}'] for s in [17, 42, 2026]]
    H1['summary'][arm] = {'in_source_mean': float(np.mean([r['in_source_select_macro_f1'] for r in rs])), 'd5_test_mean': float(np.mean([r['d5_test_macro_f1']['estimate'] for r in rs])),
                          'd5_test_sd': float(np.std([r['d5_test_macro_f1']['estimate'] for r in rs], ddof=1)), 'drop_mean': float(np.mean([r['drop_macro_f1'] for r in rs]))}
H1['summary']['shift_minus_phase1_by_seed'] = {s: H1['runs'][f'E11_maize_holdout-GH-K_shift_s{s}']['d5_test_macro_f1']['estimate'] - H1['runs'][f'E11_maize_holdout-GH-K_phase1_s{s}']['d5_test_macro_f1']['estimate'] for s in [17, 42, 2026]}
RESULTS['h1_secondary'] = H1; print(json.dumps(H1['summary'], indent=1))

In [ ]:
# 10. Results file, figures (risk-coverage on d5_test and d3c_test), zip
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (crop, sname) in zip(axes, [('maize', 'd5_test'), ('cassava', 'd3c_test')]):
    d = SETS[(crop, sname)][0]; y = np.array([LABEL_IDX[crop][l] for l in d.research_label])
    for inst in ['phase1', 'v3']:
        p = softmax(LOG[(crop, inst, sname)], INST[(crop, inst)]['T']); cov, risk = risk_coverage(p, y); ax.plot(cov, risk, label=INST[(crop, inst)]['label'])
    if crop == 'maize':
        p1 = np.load(f'{PRED}/h3_maize_d1_adapted_probs.npy'); cov, risk = risk_coverage(p1, y); ax.plot(cov, risk, linestyle='--', label=f'Phase 1 + registered adaptation ({method}, N={N})')
    ax.set_xlabel('coverage (share of images accepted)'); ax.set_ylabel('selective risk (error among accepted)'); ax.set_title(f'{crop}: {sname}'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(f'{OUT}/figure_final_risk_coverage.png', dpi=130); plt.show(); plt.close(fig)
RESULTS['environment'] = env; RESULTS['notebook'] = '11'
json.dump(RESULTS, open(f'{OUT}/final_results.json', 'w'), indent=1, default=float)
import zipfile
zp = '/content/nb11_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*'):
        if os.path.isfile(p): z.write(p, os.path.basename(p))
    for p in glob.glob(f'{PRED}/pred_*.csv'): z.write(p, f'predictions/{os.path.basename(p)}')
    for p in [f3, f3.replace('.json', '.sha256'), unseal_path, adec_path, f'{DRIVE}/records/protocol_v3_deviations.json']:
        if os.path.exists(p): z.write(p, os.path.basename(p))
from google.colab import files
files.download(zp); print('nb11_results.zip ready')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb11_results.zip`. The copies used for the published results are in this repository under `records/nb11/`.